In [8]:
%pip install matplotlib seaborn
%pip install scikit-learn matplotlib seaborn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.3 MB 299.6 kB/s eta 0:00:27
   -- ------------------------------------- 0.5/8.3 MB 299.6 kB/s eta 0:00:27
   -- ------------------------------------- 0.5/8.3 MB 299.6 kB/s eta 0:00:27
   -- ------------------------------------- 0.5/8.3 MB 299.6 kB/s eta 0:00:27
   -- ------------------------------------- 0.5/8.3 MB 299.6 kB/s eta 0:00:27
   -- ------------------------------------- 0.5/8.3 MB 299


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd

folder = r'E:\saad\intenships\Arch Technologies\Credit Card Fraud Detection'
train = pd.read_csv(folder + r'\train.csv')
test = pd.read_csv(folder + r'\test.csv')

print(train.shape, test.shape)
print(train.columns.tolist())
print(train.dtypes)
print("In train but not in test:", set(train.columns) - set(test.columns))
print(train.head())

(1296675, 23) (555719, 23)
['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']
Unnamed: 0                 int64
trans_date_trans_time     object
cc_num                     int64
merchant                  object
category                  object
amt                      float64
first                     object
last                      object
gender                    object
street                    object
city                      object
state                     object
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                       object
dob                       object
trans_num                 object
unix_time                  int64
merch_lat                float64
merch_long               float64

In [11]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, HistGradientBoostingClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, average_precision_score

folder = r'E:\saad\intenships\Arch Technologies\Credit Card Fraud Detection'
train = pd.read_csv(folder + r'\train.csv')
test = pd.read_csv(folder + r'\test.csv')

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

def prep(df):
    t = pd.to_datetime(df['trans_date_trans_time'])
    d = pd.DataFrame(index=df.index)
    d['amt'] = df['amt']
    d['log_amt'] = np.log1p(df['amt'])
    d['hour'] = t.dt.hour
    d['dayofweek'] = t.dt.dayofweek
    d['age'] = (t - pd.to_datetime(df['dob'])).dt.days / 365.25
    d['city_pop'] = df['city_pop']
    d['is_male'] = (df['gender'] == 'M').astype(int)
    d['distance_km'] = haversine_km(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
    return d.join(pd.get_dummies(df['category'], prefix='cat', dtype=int))

X_train, y_train = prep(train), train['is_fraud']
X_test, y_test = prep(test), test['is_fraud']
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"Fraud rate  train: {y_train.mean():.3%}  test: {y_test.mean():.3%}")
print("Features:", X_train.shape[1])

# --- Supervised model (the one that should actually be your main result) ---
gb = HistGradientBoostingClassifier(class_weight='balanced', random_state=1)
gb.fit(X_train, y_train)
print("\n=== Gradient Boosting (supervised) ===")
print(classification_report(y_test, gb.predict(X_test), digits=3))
print("PR-AUC:", round(average_precision_score(y_test, gb.predict_proba(X_test)[:, 1]), 4))

# --- Anomaly detectors (your original three methods) ---
scaler = StandardScaler().fit(X_train)
Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)

rng = np.random.RandomState(1)
tr_idx = rng.choice(len(Xtr), 20000, replace=False)           # slow models train on a subsample
te_idx = rng.choice(len(Xte), 100000, replace=False)          # and are scored on a test sample
fraud_rate = y_train.mean()

models = {
    "Isolation Forest": IsolationForest(n_estimators=200, contamination=fraud_rate, random_state=1),
    "Local Outlier Factor": LocalOutlierFactor(n_neighbors=20, contamination=fraud_rate, novelty=True),
    "One-Class SVM": OneClassSVM(kernel='rbf', gamma='scale', nu=fraud_rate),
}

for name, m in models.items():
    m.fit(Xtr[tr_idx])
    pred = (m.predict(Xte[te_idx]) == -1).astype(int)
    score = -m.score_samples(Xte[te_idx])
    print(f"\n=== {name} ===")
    print(classification_report(y_test.iloc[te_idx], pred, digits=3, zero_division=0))
    print("PR-AUC:", round(average_precision_score(y_test.iloc[te_idx], score), 4))

Fraud rate  train: 0.579%  test: 0.386%
Features: 22

=== Gradient Boosting (supervised) ===
              precision    recall  f1-score   support

           0      1.000     0.980     0.990    553574
           1      0.159     0.972     0.273      2145

    accuracy                          0.980    555719
   macro avg      0.579     0.976     0.631    555719
weighted avg      0.997     0.980     0.987    555719

PR-AUC: 0.8609

=== Isolation Forest ===
              precision    recall  f1-score   support

           0      0.997     0.995     0.996     99651
           1      0.154     0.281     0.199       349

    accuracy                          0.992    100000
   macro avg      0.576     0.638     0.597    100000
weighted avg      0.995     0.992     0.993    100000

PR-AUC: 0.0893

=== Local Outlier Factor ===
              precision    recall  f1-score   support

           0      0.997     0.995     0.996     99651
           1      0.075     0.120     0.092       349

   